# Validation statistics and Visualizations for CRS & training set 

In [1]:
import sys
import os

# Add the project root directory to sys.path
project_root = os.path.abspath("../../")  # Adjust the relative path as needed
if project_root not in sys.path:
    sys.path.append(project_root)

## Inspecting the CRS data

In [ ]:
# Load from feather
import pandas as pd

crs_raw = pd.read_feather("../../data/raw/crs_raw.feather")

In [ ]:
# Alphabetically sorted columns
sorted(crs_raw.columns)

crs_raw.shape

In [ ]:
# Histogram of year column with bin size = 1 year
print(crs_raw['year'].min(), crs_raw['year'].max())
crs_raw['year'].hist(bins=range(crs_raw['year'].min(), crs_raw['year'].max() + 2), edgecolor='black')

import matplotlib.pyplot as plt
plt.title("Number of projects by year")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Set ticks to start at the first year and end at the last year
plt.xticks(range(crs_raw['year'].min(), crs_raw['year'].max() + 1, 5))  # Adjust step size (e.g., 5) as needed
plt.show()


#### Evaluation of purpose code use over time 

In [ ]:
# Count of values in the purpose_code column for each year
counts_16062 = crs_raw[crs_raw['purpose_code'] == 16062].groupby('year')['purpose_code'].count()
counts_15250 = crs_raw[crs_raw['purpose_code'] == 15250].groupby('year')['purpose_code'].count()
counts_15170 = crs_raw[crs_raw['purpose_code'] == 15170].groupby('year')['purpose_code'].count()
counts_15180 = crs_raw[crs_raw['purpose_code'] == 15180].groupby('year')['purpose_code'].count()

# Plot the counts for all purpose codes
plt.figure(figsize=(12, 6))
plt.plot(counts_16062.index, counts_16062.values, label=f'16062 - Stat. cap (total {counts_16062.sum()})', marker='o')
plt.plot(counts_15250.index, counts_15250.values, label=f'15250 - mining (total {counts_15250.sum()})', marker='o')
plt.plot(counts_15170.index, counts_15170.values, label=f'15170 - womens equality (total {counts_15170.sum()})', marker='o')
plt.plot(counts_15180.index, counts_15180.values, label=f'15180 - ending violence against women (total {counts_15180.sum()})', marker='o')

# Add title, labels, and legend
plt.title('Counts of Purpose Codes by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.legend()
plt.show()

#### Long description statistics

In [ ]:
# Summary statistics for long description with number of na values
print(crs_raw["long_description"].isna().sum())
print(crs_raw["long_description"].describe())

# Mean length of long descriptions
crs_raw["long_description"].str.len().mean()

In [ ]:
import matplotlib.pyplot as plt 
# Histogram of long description lengths
pd.Series(crs_raw["long_description"].unique()).str.len().hist(bins=100)

plt.title(f"Histogram of Long Description Lengths (total unique long descriptions: {crs['long_description'].nunique()})")
plt.xlabel("Number of characters in long description")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import os 
import pandas as pd

# Save first 100 non-na long descriptions to csv in folder data/development/
if not os.path.exists("../../data/development/"):
    os.makedirs("../../data/development/")

# Convert the unique values to a pandas DataFrame and save to CSV
pd.DataFrame(crs_raw["long_description"].dropna().unique()[:100], columns=["long_description"]).to_csv(
    "../../data/development/crs_test_long_description.csv", index=False
)

#### Short description statistics

In [ ]:
# Make a histogram of the unique short_description lengths and include the number of None values
pd.Series(crs_raw["short_description"].unique()).str.len().hist(bins=80)

import matplotlib.pyplot as plt
plt.title(f"Histogram of Short Description Lengths (total unique short descriptions: {crs_raw['short_description'].nunique()}, None values: {crs_raw['short_description'].isna().sum()})")
plt.xlabel("Number of characters in short description")
plt.ylabel("Frequency")
plt.show()

#### Project title statistics

In [ ]:
# Make a histogram of the unique prject_title lengths and include the number of None values
pd.Series(crs["project_title"].unique()).str.len().hist(bins=100)

import matplotlib.pyplot as plt
plt.title(f"Histogram of Project Title Lengths (total unique project titles: {crs['project_title'].nunique()}, None values: {crs['project_title'].isna().sum()})")
plt.xlabel("Number of characters in project title")
plt.ylabel("Frequency")
plt.show()

## Training set inspection (after A - Title Pattern Matching)

In [2]:
import pandas as pd

# Load raw data
crs = pd.read_feather("../../data/raw/crs_raw.feather")

# Keep only project_title, short_description, long_description, and purpose_code columns
crs = crs[["project_title", "short_description", "long_description", "purpose_code"]]

In [15]:
# Load data after title matching
crs_matched = pd.read_feather("../../data/processed/crs_titles_matched.feather")
crs_matched_wo_stopwords = pd.read_feather("../../data/processed/crs_titles_matched_wo_stopwords.feather")

# Drop short_description and long_description columns from crs_matched
crs_matched = crs_matched.drop(columns=["short_description", "long_description"])
crs_matched_wo_stopwords = crs_matched_wo_stopwords.drop(columns=["short_description", "long_description"])

In [10]:
# Left join the two dataframes on the 'project_title' column
crs_merged = crs.merge(crs_matched, on='project_title', how='left')

In [45]:
# How many non-null values in the stat_keywords column?
print(crs_merged["stat_keywords"].notnull().sum())

# How many distinct long descriptions for non-null stat_keywords?
print("Final Stat filter: ", crs_merged[(crs_merged["stat_keywords"].notnull()) | (crs_merged["purpose_code"] == 16062) & (crs_merged["purpose_code"] != 15250) & (crs_merged["stat_blacklist"].isnull())]["long_description"].nunique())
print("Stat keywords detected: ", crs_merged[crs_merged["stat_keywords"].notnull()]["long_description"].nunique())
print("Stat purpose code: ", crs_merged[crs_merged["purpose_code"] == 16062]["long_description"].nunique())
print("Mining purpose code: ", crs_merged[crs_merged["purpose_code"] == 15250]["long_description"].nunique())
print("Stat blacklist: ", crs_merged[(crs_merged["stat_blacklist"].notnull())]["long_description"].nunique())

27514
Final Stat filter:  10479
Stat keywords detected:  7750
Stat purpose code:  4390
Mining purpose code:  4920
Stat blacklist:  1703


In [12]:
crs_matched.columns

Index(['project_title', 'normalized_title', 'language_x', 'lemmatized_title',
       'stat_keywords', 'stat_blacklist', 'gen_keywords', 'ai_keywords',
       'stat_acronyms', 'gen_acronyms', 'ai_acronyms', 'language_y',
       'lemmatized_title_wo_stopwords', 'stat_keywords_wo_stopwords',
       'stat_blacklist_wo_stopwords', 'gen_keywords_wo_stopwords',
       'ai_keywords_wo_stopwords', 'stat_acronyms_wo_stopwords',
       'gen_acronyms_wo_stopwords', 'ai_acronyms_wo_stopwords'],
      dtype='object')

In [16]:
# Comparison with and without stopwords 
# Drop normalized_title from crs_matched_wo_stopwords
crs_matched_wo_stopwords = crs_matched_wo_stopwords.drop(columns=["normalized_title", "language"])

# Rename the columns of the crs_matched_wo_stopwords dataframe
crs_matched_wo_stopwords = crs_matched_wo_stopwords.rename(columns={"stat_keywords": "stat_keywords_wo_stopwords", "stat_blacklist": "stat_blacklist_wo_stopwords", "gen_keywords": "gen_keywords_wo_stopwords", "ai_keywords": "ai_keywords_wo_stopwords", "stat_acronyms": "stat_acronyms_wo_stopwords", "gen_acronyms": "gen_acronyms_wo_stopwords", "ai_acronyms": "ai_acronyms_wo_stopwords", "lemmatized_title": "lemmatized_title_wo_stopwords"})

# Merge the two dataframes on the 'project_title' column
crs_matched = crs_matched.merge(crs_matched_wo_stopwords, on='project_title', how='left')

# Make xlsx with all non-null values in the columns that are either project_title, normalized_title, language, or lemmatized_title
crs_reduced = crs_matched[
    (crs_matched['stat_keywords'].notna()) |
    (crs_matched['stat_blacklist'].notna()) |
    (crs_matched['gen_keywords'].notna()) |
    (crs_matched['ai_keywords'].notna()) |
    (crs_matched['stat_acronyms'].notna()) |
    (crs_matched['gen_acronyms'].notna()) |
    (crs_matched['ai_acronyms'].notna()) |
    (crs_matched['stat_keywords_wo_stopwords'].notna()) |
    (crs_matched['stat_blacklist_wo_stopwords'].notna()) |
    (crs_matched['gen_keywords_wo_stopwords'].notna()) |
    (crs_matched['ai_keywords_wo_stopwords'].notna()) |
    (crs_matched['stat_acronyms_wo_stopwords'].notna()) |
    (crs_matched['gen_acronyms_wo_stopwords'].notna()) |
    (crs_matched['ai_acronyms_wo_stopwords'].notna())
]

# Save the filtered dataframe to an Excel file
crs_reduced.to_excel("../../data/development/crs_stopwords_comparison.xlsx", index=False)

#### Language distribution

In [ ]:
# For every language, how many projects have a non-empty stat_keywords?
language_distr = crs_matched[crs_matched["stat_keywords"].notna()].groupby("language")["stat_keywords"].count()

total = language_distr.sum()

# Remove languages with value < 2
language_distr = language_distr[language_distr >= 2]

# Sort the language distribution in descending order
language_distr = language_distr.sort_values(ascending=False)

# Make a bar plot of the language distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
bars = plt.bar(language_distr.index, language_distr.values, color='skyblue')
plt.title('Number titles matched for statistics with stopwords (total titles matched: {})'.format(total))
plt.xlabel('Language')
plt.ylabel('Number of Projects')
plt.tight_layout()

# Add values on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(height), ha='center', va='bottom', fontsize=9)

plt.show()

In [ ]:
# For every language, how many projects have a non-empty stat_keywords?
language_distr = crs_matched[crs_matched["gen_keywords"].notna()].groupby("language")["gen_keywords"].count()

total = language_distr.sum()

# Remove languages with value < 2
language_distr = language_distr[language_distr >= 2]

# Sort the language distribution in descending order
language_distr = language_distr.sort_values(ascending=False)

# Make a bar plot of the language distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
bars = plt.bar(language_distr.index, language_distr.values, color='brown')
plt.title('Number titles matched for gender with stopwords (total titles matched: {})'.format(total))
plt.xlabel('Language')
plt.ylabel('Number of Projects')
plt.tight_layout()

# Add values on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(height), ha='center', va='bottom', fontsize=9)

plt.show()

#### Keyword analysis

In [17]:
from src.text_processing import process_keywords

keywords_file_path = "../../data/keywords/review/keyword_review_modernization.xlsx"
stat_keywords = pd.read_excel(keywords_file_path, sheet_name="statistics")
gen_keywords = pd.read_excel(keywords_file_path, sheet_name="gender")

stat_keywords = process_keywords(stat_keywords, remove_stopwords=False)
gen_keywords = process_keywords(gen_keywords, remove_stopwords=False)

c:\Users\jojoa\OneDrive\Dokumente\OECD\RAnalysis\PRESS\src\text_processing.py:281: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  keywords_df = pd.concat([keywords_df, new_values_df], ignore_index=True)


In [ ]:
matched_keywords_frquency = crs_matched['stat_keywords'].explode().value_counts()

# Map each keyword to its corresponding language
keyword_language_map = {}
for col in ['en', 'fr', 'es', 'de']:
    for keyword in stat_keywords[col].dropna():
        keyword_language_map[keyword] = col

# Assign colors to each language
language_colors = {'en': 'steelblue', 'fr': 'tomato', 'es': 'mediumseagreen', 'de': 'goldenrod'}

# Get colors for each keyword in the frequency data
colors = matched_keywords_frquency.index.map(lambda x: language_colors.get(keyword_language_map.get(x, 'en'), 'gray'))

# Adjust the y-axis limits to remove extra white space
fig, ax = plt.subplots(figsize=(10, 21))
bars = ax.barh(matched_keywords_frquency.index, matched_keywords_frquency.values, color=colors)

plt.title(f"Matched Keywords in Project Titles (Total keywords matched: {matched_keywords_frquency.sum()})")
plt.xlabel("Frequency")
plt.ylabel("Matched Keywords")

# Add frequency values to the right of each bar with matching colors
for i, (v, color) in enumerate(zip(matched_keywords_frquency, colors)):
    ax.text(v + 1, i, str(v), va='center', fontsize=9, color=color)

# Color the y-tick labels to match the bar colors
ax.set_yticks(range(len(matched_keywords_frquency)))
ax.set_yticklabels(matched_keywords_frquency.index, fontsize=9, color='black')
for tick_label, color in zip(ax.get_yticklabels(), colors):
    tick_label.set_color(color)

# Adjust y-axis limits to remove extra white space
ax.set_ylim(-0.5, len(matched_keywords_frquency) - 0.5)

# Add a legend for the colors
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=color, lw=4, label=lang) for lang, color in language_colors.items()]
plt.legend(handles=legend_elements, title="Languages", loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
matched_keywords_frquency = crs_matched['gen_keywords'].explode().value_counts()

# Map each keyword to its corresponding language
keyword_language_map = {}
for col in ['en', 'fr', 'es', 'de']:
    for keyword in gen_keywords[col].dropna():
        keyword_language_map[keyword] = col

# Assign colors to each language
language_colors = {'en': 'steelblue', 'fr': 'tomato', 'es': 'mediumseagreen', 'de': 'goldenrod'}

# Get colors for each keyword in the frequency data
colors = matched_keywords_frquency.index.map(lambda x: language_colors.get(keyword_language_map.get(x, 'en'), 'gray'))

# Adjust the y-axis limits to remove extra white space
fig, ax = plt.subplots(figsize=(10, 14))
bars = ax.barh(matched_keywords_frquency.index, matched_keywords_frquency.values, color=colors)

plt.title(f"Matched Keywords in Project Titles (Total keywords matched: {matched_keywords_frquency.sum()})")
plt.xlabel("Frequency")
plt.ylabel("Matched Keywords")

# Add frequency values to the right of each bar with matching colors
for i, (v, color) in enumerate(zip(matched_keywords_frquency, colors)):
    ax.text(v + 1, i, str(v), va='center', fontsize=9, color=color)

# Color the y-tick labels to match the bar colors
ax.set_yticks(range(len(matched_keywords_frquency)))
ax.set_yticklabels(matched_keywords_frquency.index, fontsize=9, color='black')
for tick_label, color in zip(ax.get_yticklabels(), colors):
    tick_label.set_color(color)

# Adjust y-axis limits to remove extra white space
ax.set_ylim(-0.5, len(matched_keywords_frquency) - 0.5)

# Add a legend for the colors
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=color, lw=4, label=lang) for lang, color in language_colors.items()]
plt.legend(handles=legend_elements, title="Languages", loc="upper right")

plt.tight_layout()
plt.show()